# Sugarcane Yield Prediction — Early-Stage EDA & Preprocessing

**Goal:** Predict `Yield_Quintal_per_Acre` using only information available **at or before planting** — weather (seasonal averages), soil test results, field/plot parameters, irrigation setup, and planting decisions.

**Why "early-stage only"?** Columns like `Cane_Height_cm`, `Brix_Value`, `Germination_%`, `Disease_Type`, `Growth_Stage`, `Harvesting_Date`, etc. are only known *during or after* the crop has grown — using them would be data leakage for a genuinely early prediction model (you'd be using information that doesn't exist yet at prediction time).

This notebook covers:
1. Load & inspect data
2. Define early-stage feature set (drop leakage + ID/constant columns)
3. Missing value analysis
4. Target distribution
5. Numeric feature correlations with yield
6. Categorical feature relationships with yield
7. Preprocessing pipeline (impute + encode)
8. Save cleaned, model-ready dataset


https://www.kaggle.com/datasets/priyankabarik/sugar-cane-prediction-dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('D:\SIT\Sem.5\PBL V\Datasets\FINAL_SUGARCANE_DATASET.csv')
print(df.shape)
df.head()

## 1. Basic info & data quality check

In [ ]:
df.info()

In [ ]:
# Columns with only 1 unique value carry zero predictive signal -- drop them
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
print("Constant columns (no variation):", constant_cols)

## 2. Define the early-stage feature set

These are features that a farmer / agronomist would know **before or at planting time**:
weather (typically forecast/seasonal normals), soil test results, field geometry, irrigation
infrastructure, and planting-method decisions.

Excluded (leakage or unusable):
- Growth/harvest measurements: `Germination_%`, `Tillering_Count`, `Cane_Height_cm`, `Cane_Diameter_cm`,
  `Internode_Length_cm`, `Brix_Value`, `Growth_Stage`, `Harvesting_Date`, `Crop_Duration_Days`
- In-season management/observations: `Disease_Type`, `Disease_Severity`, `Pest_Level`, `Pesticide_Usage`,
  `Fertilizer_Type`, `Fertilizer_Quantity`, `Application_Timing`, `Fertilizer_Split`, `Machinery_Use`, `Labor_Input`
- Constant / non-informative: `State`, `District`, `Sugar_Mill`, `Region`, `Khasra_No`, `Planting_Date`
  (raw date; `Month`/`Season`/`Year` already capture the seasonal signal)


In [ ]:
early_stage_features = [
    'Month', 'Temp_Min_C', 'Temp_Max_C', 'Temp_Avg_C', 'Rainfall_Total_mm', 'Rainfall_Seasonal_mm',
    'Humidity_%', 'Solar_Radiation_MJ_m2_day', 'Sunshine_Hours_hh_mm', 'Wind_Speed_kmph',
    'Evapotranspiration_mm_day', 'Dew_Point_C', 'Heat_Stress_Days', 'Frost_Days',
    'Soil_Type', 'Soil_pH', 'EC', 'Organic_Carbon_%', 'Soil_Moisture_%', 'Sand_%', 'Silt_%', 'Clay_%',
    'Soil_Depth_cm', 'Water_Holding_Capacity_%', 'Nitrogen_kg_per_acre', 'Phosphorus_kg_per_acre',
    'Potassium_kg_per_acre', 'Zinc_mg_per_kg', 'Iron_mg_per_kg', 'Copper_mg_per_kg', 'Manganese_mg_per_kg',
    'Sulfur_kg_per_acre', 'Irrigation_Frequency_Level', 'Water_Quantity_liters_per_acre',
    'Irrigation_Method_Type', 'Groundwater_Level_meters', 'Water_Quality_Category',
    'Seed_Rate', 'Row_Spacing_cm', 'Row_Gap_cm', 'Plant_Density', 'Intercropping', 'Weed_Control',
    'Soil_Condition_At_Planting', 'Planting_Method', 'Mulching', 'Drainage_Condition', 'Variety',
    'Latitude', 'Longitude', 'Altitude_m', 'Tehsil', 'Agro_Cluster', 'Season', 'Year'
]
target = 'Yield_Quintal_per_Acre'

data = df[early_stage_features + [target]].copy()
print("Working dataset shape:", data.shape)
data.head()

In [ ]:
# Sunshine_Hours_hh_mm is stored as "H:MM" text -- convert to decimal hours
def hhmm_to_hours(val):
    if pd.isna(val):
        return np.nan
    try:
        h, m = str(val).split(':')
        return int(h) + int(m) / 60
    except Exception:
        return np.nan

data['Sunshine_Hours'] = data['Sunshine_Hours_hh_mm'].apply(hhmm_to_hours)
data.drop(columns=['Sunshine_Hours_hh_mm'], inplace=True)
data['Sunshine_Hours'].describe()

## 3. Missing value analysis

In [ ]:
missing_pct = (data.isnull().mean() * 100).round(1).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
print(missing_pct)

plt.figure(figsize=(8, 10))
sns.barplot(x=missing_pct.values, y=missing_pct.index, color='steelblue')
plt.xlabel('% missing')
plt.title('Missing values by feature (early-stage set)')
plt.tight_layout()
plt.show()

No column is missing more than ~5-6%, so we can safely impute (median for numeric,
mode for categorical) rather than dropping rows or columns.

## 4. Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data[target], kde=True, ax=axes[0], color='seagreen')
axes[0].set_title('Yield distribution (Quintal/Acre)')
sns.boxplot(x=data[target], ax=axes[1], color='seagreen')
axes[1].set_title('Yield boxplot')
plt.tight_layout()
plt.show()

print(data[target].describe())

## 5. Numeric feature correlations with yield

In [ ]:
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
corr = data[numeric_cols].corr()[target].drop(target).sort_values(key=abs, ascending=False)
print(corr)

plt.figure(figsize=(8, 10))
sns.barplot(x=corr.values, y=corr.index, hue=corr.index, palette='vlag', legend=False)
plt.title('Correlation of numeric features with Yield')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(data[numeric_cols].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap -- early-stage numeric features')
plt.tight_layout()
plt.show()

## 6. Categorical features vs yield

In [ ]:
cat_cols = data.select_dtypes(include='object').columns.tolist()
print(cat_cols)

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(cat_cols[:9]):
    sns.boxplot(data=data, x=col, y=target, ax=axes[i])
    axes[i].set_title(f'{col} vs Yield')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 7. Preprocessing pipeline (impute + encode)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = data.drop(columns=[target])
y = data[target]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print(f"Numeric features ({len(numeric_features)}):", numeric_features)
print(f"Categorical features ({len(categorical_features)}):", categorical_features)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
# Fit-transform on train, transform on test (prevents leakage from test set stats)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print("Processed shape:", X_train_processed.shape)
print("Total features after encoding:", len(feature_names))

## 8. Save cleaned dataset for the modeling notebook

In [ ]:
import joblib

# Save the raw (unencoded) train/test splits -- model notebook will reuse `preprocessor`
X_train.to_csv('X_train_raw.csv', index=False)
X_test.to_csv('X_test_raw.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
joblib.dump(preprocessor, 'preprocessor.joblib')

print("Saved: X_train_raw.csv, X_test_raw.csv, y_train.csv, y_test.csv, preprocessor.joblib")

### Next step
With `X_train_raw.csv` / `X_test_raw.csv` / the fitted `preprocessor` saved, the next notebook will
train and compare models (Linear Regression, Ridge, Decision Tree, Random Forest, Gradient Boosting)
on this early-stage feature set and evaluate with RMSE / MAE / R².